# log-samples-eval-callback composite — cx30: Trainer wires wandb.init + every-K-step log_samples callback that emits wandb.Image

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `wandb-init-run`, `log-samples-eval-callback`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F
import wandb
from torch.utils.data import DataLoader, TensorDataset

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "log-samples-eval-callback"
DD_ATOM_IDS = ["wandb-init-run", "log-samples-eval-callback"]
DD_SUBTOPICS = ["Logging: wandb.init run", "Logging: log-samples eval callback"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Visual generative models (VAEs, GANs, diffusion) log SAMPLES periodically so you can SEE training progress in the wandb UI. The composition:

1. **wandb-init-run** — open one run at start of training.
2. **log-samples-eval-callback** — every K steps, generate N samples from the model, wrap each as a `wandb.Image(...)`, and log them with `wandb.log({'samples': [Image, Image, ...]})`. The eval-callback cadence is `step % eval_every == 0`.

**Anatomy.**
```python
class SampleLoggingTrainer:
    def pre_training_setup(self):
        wandb.init(project=..., name=..., config=self.args)        # wandb-init-run.
    def training_step(self, batch):
        ...
        self.step += 1
        if self.step % self.args.eval_every == 0:
            self.log_samples()                                      # log-samples-eval-callback.
    def log_samples(self):
        with t.inference_mode():
            samples = self.model.sample(n=self.args.n_eval)
        images = [wandb.Image(s) for s in samples]
        wandb.log({'samples': images}, step=self.step)
```

**Step 0 vs step K.** Some recipes log samples at step 0 too (initial random model baseline). The convention varies; here we fire when `step > 0 and step % eval_every == 0` — so for `eval_every=3, n_steps=10` we fire at steps 3, 6, 9 (3 fires). The test pins down exactly this cadence.

### Composite Exercise — Trainer wires wandb.init + every-K-step log_samples callback that emits wandb.Image

**Atoms exercised together**: `wandb-init-run`, `log-samples-eval-callback`

Implement `cx30_make_sample_logger_trainer()` returning a `SampleLoggingTrainer` class.

Required structure:
- `SampleLoggingTrainer.__init__(self, model, optimizer, loss_fn, train_loader, args)`:
  - Store all five. `self.step = 0`. `args` has `.wandb_project`, `.wandb_name`, `.lr`, `.epochs`, `.eval_every`, `.n_eval`.
- `SampleLoggingTrainer.pre_training_setup(self)`:
  - `wandb.init(project=args.wandb_project, name=args.wandb_name, config=args)` (atom: wandb-init-run).
- `SampleLoggingTrainer.training_step(self, batch)`:
  - Forward → scalar loss → zero_grad → backward → opt.step → `self.step += 1`.
  - If `self.step > 0 and self.step % self.args.eval_every == 0`, call `self.log_samples()` (atom: log-samples-eval-callback).
  - Return scalar loss.
- `SampleLoggingTrainer.log_samples(self)`:
  - Create `self.args.n_eval` dummy sample tensors: `[t.randn(3, 8, 8) for _ in range(self.args.n_eval)]` (CHW shape — wandb.Image accepts that).
  - Wrap each as `wandb.Image(s)`.
  - `wandb.log({'samples': images}, step=self.step)`.
- `SampleLoggingTrainer.fit(self, n_epochs)`:
  - `pre_training_setup()` once, then loop epochs × loader, calling `training_step(batch)`.

The test verifies (with mocked wandb):
- `wandb.init` called once with project/name/config.
- `wandb.Image` constructor called `n_eval` times PER firing of the callback.
- `wandb.log` called with `{'samples': [...]}` at exactly the expected step counts.
- The number of fires matches `n_steps_total // eval_every` (since we skip step 0).

In [ ]:
def cx30_make_sample_logger_trainer():
    import sys
    from unittest.mock import MagicMock
    sys.modules.setdefault('wandb', MagicMock())
    import wandb

    class SampleLoggingTrainer:
        def __init__(self, model, optimizer, loss_fn, train_loader, args):
            self.model = model
            self.optimizer = optimizer
            self.loss_fn = loss_fn
            self.train_loader = train_loader
            self.args = args
            self.step = 0

        def pre_training_setup(self):
            # Atom A (wandb-init-run).
            wandb.init(
                project=self.args.wandb_project,
                name=self.args.wandb_name,
                config=self.args,
            )

        def training_step(self, batch):
            x, y = batch
            logits = self.model(x)
            loss = self.loss_fn(logits, y)
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            self.step += 1
            # Atom B (log-samples-eval-callback): every-K-steps cadence (skip step 0).
            if self.step > 0 and self.step % self.args.eval_every == 0:
                self.log_samples()
            return loss

        def log_samples(self):
            # Generate n_eval dummy samples (CHW so wandb.Image is happy).
            with t.inference_mode():
                samples = [t.randn(3, 8, 8) for _ in range(self.args.n_eval)]
            images = [wandb.Image(s) for s in samples]
            wandb.log({'samples': images}, step=self.step)

        def fit(self, n_epochs):
            self.pre_training_setup()
            for _ in range(n_epochs):
                for batch in self.train_loader:
                    self.training_step(batch)

    return SampleLoggingTrainer


<details><summary>Show solution — cx30</summary>

```python
def cx30_make_sample_logger_trainer():
    import sys
    from unittest.mock import MagicMock
    sys.modules.setdefault('wandb', MagicMock())
    import wandb

    class SampleLoggingTrainer:
        def __init__(self, model, optimizer, loss_fn, train_loader, args):
            self.model = model
            self.optimizer = optimizer
            self.loss_fn = loss_fn
            self.train_loader = train_loader
            self.args = args
            self.step = 0

        def pre_training_setup(self):
            # Atom A (wandb-init-run).
            wandb.init(
                project=self.args.wandb_project,
                name=self.args.wandb_name,
                config=self.args,
            )

        def training_step(self, batch):
            x, y = batch
            logits = self.model(x)
            loss = self.loss_fn(logits, y)
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            self.step += 1
            # Atom B (log-samples-eval-callback): every-K-steps cadence (skip step 0).
            if self.step > 0 and self.step % self.args.eval_every == 0:
                self.log_samples()
            return loss

        def log_samples(self):
            # Generate n_eval dummy samples (CHW so wandb.Image is happy).
            with t.inference_mode():
                samples = [t.randn(3, 8, 8) for _ in range(self.args.n_eval)]
            images = [wandb.Image(s) for s in samples]
            wandb.log({'samples': images}, step=self.step)

        def fit(self, n_epochs):
            self.pre_training_setup()
            for _ in range(n_epochs):
                for batch in self.train_loader:
                    self.training_step(batch)

    return SampleLoggingTrainer
```

Wrapping each sample in `wandb.Image` (rather than logging raw tensors) tells wandb to render it as an image in the run UI — wandb accepts numpy arrays, PIL Images, torch tensors in CHW format, and matplotlib figures. The list-of-Images value under `'samples'` becomes a gallery in the dashboard. Logging samples is expensive on real wandb (network upload), so keep `n_eval` small (≤16) and `eval_every` large (≥100 in real training).
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx30'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx30',
        'subtopics': ["Logging: wandb.init run", "Logging: log-samples eval callback"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()